# MTC Smart Challenge 2026 — Inferencia y Submission (Fase 2.2)

Notebook **autocontenido** (no necesita git clone). Produce `submission.csv`.

**Inputs que deben estar adjuntos en /kaggle/input:**
- Dataset con `best.pt` (output del notebook de entrenamiento)
- Dataset con los frames de test (`.jpg`)
- Dataset de la competencia con `sample_submission.csv`

**Qué hace este notebook:**
1. Descubre `best.pt`, frames de test y `sample_submission.csv` con `rglob` (sin slugs hardcodeados).
2. Construye un índice `{frame_id → path}` de todos los JPG en `/kaggle/input`.
3. Inferencia OBB con `conf=0.01` (alto recall), `iou_nms=0.6`, `imgsz=1024`.
4. Formatea cada detección como `score category_id cx cy width height angle_deg`
   separado por `;`; `none` si no hay detecciones.
5. Capea a top-300 por frame; descarta detecciones inválidas.
6. Respeta exactamente los 23,579 IDs y el orden de `sample_submission.csv`.
7. Escribe `/kaggle/working/submission.csv` con sanity checks.

In [ ]:
# ── Celda 1: Instalación y GPU ────────────────────────────────────────────
# ultralytics viene preinstalado en Kaggle GPU kernels; pip actualiza si hace falta
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"])

import math, os
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

WORK  = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

# ── Parámetros de inferencia ──────────────────────────────────────────────
CONF    = 0.01   # umbral bajo → máximo recall; la métrica AP tolera FP si los scores
                 # están bien ordenados; filtrar aquí solo perjudicaría el ranking
IOU_NMS = 0.6    # NMS; más alto que el default (0.45) para no suprimir vehículos próximos
IMGSZ   = 1024   # Ultralytics reescala las predicciones al tamaño original automáticamente
MAX_DET = 300    # límite por frame, ordenado por score DESC

n_gpu  = torch.cuda.device_count()
device = 0 if n_gpu >= 1 else "cpu"
print(f"PyTorch {torch.__version__}  |  GPUs: {n_gpu}  |  device: {device}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB")
print(f"conf={CONF}  iou_nms={IOU_NMS}  imgsz={IMGSZ}  max_det={MAX_DET}")

In [ ]:
# ── Celda 2: Descubrimiento de artefactos (sin slugs hardcodeados) ────────

# ── best.pt ───────────────────────────────────────────────────────────────
best_candidates = sorted(
    INPUT.rglob("best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,   # el más reciente primero si hay varios
)
assert best_candidates, (
    "No se encontro best.pt en /kaggle/input. "
    "Adjunta el dataset de outputs del notebook de entrenamiento."
)
if len(best_candidates) > 1:
    print(f"AVISO: {len(best_candidates)} archivos best.pt — usando el mas reciente:")
    for c in best_candidates:
        print(f"  {c}  ({c.stat().st_size / 1e6:.1f} MB)")
BEST_PT = best_candidates[0]
print(f"best.pt      : {BEST_PT}")
print(f"             : {BEST_PT.stat().st_size / 1e6:.1f} MB")

# ── sample_submission.csv ─────────────────────────────────────────────────
sample_candidates = sorted(INPUT.rglob("sample_submission.csv"))
assert sample_candidates, "No se encontro sample_submission.csv en /kaggle/input."
SAMPLE_CSV = sample_candidates[0]
print(f"sample_csv   : {SAMPLE_CSV}")

sample_df = pd.read_csv(SAMPLE_CSV)
assert "Id" in sample_df.columns, "sample_submission.csv no tiene columna 'Id'"
test_ids = sample_df["Id"].tolist()
print(f"Test IDs     : {len(test_ids):,}  (esperados: 23,579)")

# ── Índice de frames: stem → path ─────────────────────────────────────────
# Escanea TODOS los JPG en /kaggle/input; el stem (nombre sin extensión) es el frame_id
print("\nIndexando frames en /kaggle/input (puede tardar ~30 s) ...")
frames_lookup: dict = {}
for p in INPUT.rglob("*.jpg"):
    frames_lookup[p.stem] = p
print(f"JPGs indexados: {len(frames_lookup):,}")

found   = [fid for fid in test_ids if fid in frames_lookup]
missing = [fid for fid in test_ids if fid not in frames_lookup]
print(f"Test encontrados: {len(found):,} / {len(test_ids):,}")
if missing:
    print(f"Test faltantes  : {len(missing):,} — primeros 5: {missing[:5]}")
    if len(missing) > len(test_ids) * 0.05:
        raise AssertionError(
            f"Faltan {len(missing):,} frames ({100*len(missing)/len(test_ids):.1f}%). "
            "Verifica que el dataset de frames esta adjunto."
        )
    print("  (AVISO: frames faltantes seran emitidos como 'none')")

# ── Diagnóstico de datasets adjuntos ──────────────────────────────────────
print("\nDatasets adjuntos en /kaggle/input:")
for d in sorted(INPUT.iterdir()):
    if d.is_dir():
        n_files = sum(1 for _ in d.rglob("*") if _.is_file())
        print(f"  {d.name:<50}  {n_files:>7} archivos")

In [ ]:
# ── Celda 3: Inferencia YOLO-OBB ─────────────────────────────────────────
#
# Ultralytics devuelve result.obb.xywhr → (cx, cy, w, h, angle_rad) en PÍXELES
# del tamaño original de la imagen (rescalado automático desde imgsz=1024).
# angle_rad en radianes → convertir a grados y normalizar a [0, 360).
# category_id = cls (0-indexed) + 1  →  rango [1, 9] del concurso.
#
# Estrategia de memoria: stream=True procesa una imagen a la vez en el generador;
# Path(result.path).stem identifica el frame_id sin depender del orden.

model = YOLO(str(BEST_PT))
print(f"Modelo cargado: {BEST_PT.name}")

predictions: dict = {}

# Frames sin imagen disponible → None directo (se formatearán como 'none')
for fid in missing:
    predictions[fid] = []

found_paths = [str(frames_lookup[fid]) for fid in found]

if not found_paths:
    print("AVISO: no hay frames de test para inferir.")
else:
    n_invalid = 0

    for result in tqdm(
        model.predict(
            source=found_paths,
            conf=CONF,
            iou=IOU_NMS,
            imgsz=IMGSZ,
            device=device,
            verbose=False,
            stream=True,
            max_det=MAX_DET,
        ),
        total=len(found_paths),
        desc="Inferencia",
    ):
        fid  = Path(result.path).stem   # frame_id desde el nombre del archivo
        dets = []

        if result.obb is not None and len(result.obb):
            xywhr = result.obb.xywhr.cpu().numpy()   # (N, 5): píxeles imagen original
            confs = result.obb.conf.cpu().numpy()
            clses = result.obb.cls.cpu().numpy().astype(int)

            for i in range(len(xywhr)):
                cx       = float(xywhr[i, 0])
                cy       = float(xywhr[i, 1])
                w        = float(xywhr[i, 2])
                h        = float(xywhr[i, 3])
                ang_rad  = float(xywhr[i, 4])
                score    = float(confs[i])
                cat_id   = int(clses[i]) + 1          # 0-indexed → 1-indexed
                ang_deg  = math.degrees(ang_rad) % 360

                # ── Validación: descartar detecciones inválidas ───────────────
                if not (w > 0 and h > 0):
                    n_invalid += 1
                    continue
                if not (1 <= cat_id <= 9):
                    n_invalid += 1
                    continue
                if not all(math.isfinite(v) for v in (score, cx, cy, w, h, ang_deg)):
                    n_invalid += 1
                    continue

                dets.append((score, cat_id, cx, cy, w, h, ang_deg))

        # Ordenar por score DESC y capear a MAX_DET
        dets.sort(key=lambda x: -x[0])
        predictions[fid] = dets[:MAX_DET]

    n_with = sum(1 for v in predictions.values() if v)
    n_none = sum(1 for v in predictions.values() if not v)
    print(f"\nDetecciones invalidas descartadas : {n_invalid:,}")
    print(f"Frames con >=1 deteccion         : {n_with:,}")
    print(f"Frames sin detecciones ('none')  : {n_none:,}")

In [ ]:
# ── Celda 4: Formatear, escribir submission.csv y sanity checks ───────────

def format_row(dets: list) -> str:
    """Lista de tuplas → string de submission (o 'none' si vacío).

    Formato por detección: 'score category_id cx cy width height angle_deg'
    Separador entre detecciones: ';'
    Nunca retorna string vacío.
    """
    if not dets:
        return "none"
    parts = [
        f"{score:.4f} {cat_id} {cx:.2f} {cy:.2f} {w:.2f} {h:.2f} {ang:.2f}"
        for score, cat_id, cx, cy, w, h, ang in dets
    ]
    return ";".join(parts)

# Construir en el ORDEN EXACTO de sample_submission.csv
targets = [format_row(predictions.get(fid, [])) for fid in test_ids]
submission_df = pd.DataFrame({"Id": test_ids, "Target": targets})

OUT_CSV = WORK / "submission.csv"
submission_df.to_csv(OUT_CSV, index=False)

# ── Sanity checks ─────────────────────────────────────────────────────────
errors = []

if len(submission_df) != len(sample_df):
    errors.append(f"Filas: {len(submission_df):,} != {len(sample_df):,} esperadas")

if list(submission_df["Id"]) != list(sample_df["Id"]):
    # Localizar primer desajuste
    for i, (a, b) in enumerate(zip(submission_df["Id"], sample_df["Id"])):
        if a != b:
            errors.append(f"Orden de IDs difiere en posicion {i}: '{a}' vs '{b}'")
            break

if submission_df["Target"].isna().any():
    n_na = submission_df["Target"].isna().sum()
    errors.append(f"Target tiene {n_na} celdas NaN")

if (submission_df["Target"] == "").any():
    n_empty = (submission_df["Target"] == "").sum()
    errors.append(f"Target tiene {n_empty} celdas con string vacio")

if errors:
    print("ERRORES EN SUBMISSION:")
    for e in errors:
        print(f"  {e}")
    raise ValueError("Submission invalida — corregir errores arriba antes de subir")

# ── Reporte final ─────────────────────────────────────────────────────────
n_none_final = (submission_df["Target"] == "none").sum()
n_det_final  = len(submission_df) - n_none_final

print(f"Submission guardada : {OUT_CSV}")
print(f"Tamanio             : {OUT_CSV.stat().st_size / 1e3:.0f} KB")
print()
print("=" * 48)
print("RESUMEN SUBMISSION")
print("=" * 48)
print(f"  Filas totales       : {len(submission_df):,} / {len(sample_df):,}")
print(f"  Con detecciones     : {n_det_final:,}  ({100*n_det_final/len(submission_df):.1f}%)")
print(f"  Sin detecciones     : {n_none_final:,}  ({100*n_none_final/len(submission_df):.1f}%)")
print("=" * 48)
print("Todas las sanity checks OK.")
print()
print("Primeras 3 filas:")
for _, row in submission_df.head(3).iterrows():
    tgt = row["Target"]
    preview = tgt[:120] + "..." if len(tgt) > 120 else tgt
    print(f"  {row['Id']}  ->  {preview}")